# 03 — Prompt-Invariant Comparison Sources

Deep-dive into the four non-LLM translation sources. This notebook uses “baseline” as shorthand for **prompt-invariant comparison source**, but the four sources do not have the same epistemic status:

- **Community-curated reference** (1 source): **Wikipedia** interlanguage links — what editors in each language community have attached to the `Digital Humanities` page. This is not machine translation; it is a human-negotiated equivalence mapping. The category is extensible: future additions could include IATE, LCSH, DH journal subject headings, or Wikidata.
- **Machine translation baselines** (3 sources): **Google Translate**, **EasyNMT** (Helsinki-NLP opus-mt), and **Lingvanex** — algorithmic translations produced by neural MT systems trained on parallel corpora.

A note on naming: the **Wikimedia language codes** described in nb01 §1.1 are the *registry* used to define the pipeline language set. The **Wikipedia translations** used here are the per-article interlanguage-link strings keyed off those codes — same identifiers, different data layer.

These two categories answer different questions. The MT baselines tell us how current MT systems handle DH terminology. Wikipedia tells us what some language communities have already stabilized as a page title or interlanguage-link label. LLM outputs sit between these poles and are handled in Notebook 04.

Because these sources run once per language with no prompt variation, they are a cleaner lens for several distinct questions:

- **Coverage**: which language families can each source reach, and where are the structural gaps?
- **Overlap**: how many pipeline languages have translations from multiple sources, and how many have none at all?
- **Fidelity**: when a source produces output, does it translate the term or echo `Digital Humanities` unchanged?
- **Automated review signals**: which translation-level signals fire for each source?
- **Filtering funnel**: what remains under full, quality-filtered, and search-ready views?

**Sections**
1. Coverage by source and language family, including EasyNMT structural gaps and overlap
2. Baseline output shape: word-count distribution without prompt variation
3. Pass-through rate: sources that echo the source term unchanged
4. Automated review signals per source
5. Mixed-script output in prompt-invariant translations
6. Filter funnel: nominal coverage → quality-filtered → search-ready


In [1]:
import os
import sys
from collections import defaultdict

import pandas as pd
import altair as alt
from pathlib import Path

alt.data_transformers.enable("vegafusion")

sys.path.insert(0, str(Path("..").resolve()))
from scripts.utils import get_data_directory_path, read_csv_file, get_language_family
from scripts.exploration.explore_confidence_within_variant import load_variant_df
from scripts.exploration.translation_classifier import (
    curate_translation,
    has_source_leakage,
    is_placeholder_term,
    is_repetition_loop,
    has_extreme_term_length,
    has_unicode_escape,
)

DATA_DIR = get_data_directory_path()
TERM = "Digital Humanities"
TERM_SLUG = TERM.lower().replace(" ", "_")

BASELINE_SERVICES = {
    "Wikipedia": "wikipedia_translated_term",
    "Google Translate": "gt_translated_term",
    "EasyNMT": "enmt_translated_term",
    "Lingvanex": "lingvanex_translated_term",
}
LLM_SERVICES = {
    "OpenAI":   "openai_translated_term",
    "Claude":   "claude_translated_term",
    "Gemini":   "gemini_translated_term",
    "DeepSeek": "deepseek_translated_term",
    "Llama":    "llama_translated_term",
    "Gemma":    "gemma_translated_term",
    "Qwen":     "qwen_translated_term",
    "Mistral":  "mistral_translated_term",
}
ALL_SERVICES = {**BASELINE_SERVICES, **LLM_SERVICES}

print(f"Data directory: {DATA_DIR}")

Retrieving translation pipeline data directory path...

Data directory: /Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets


In [2]:
# ── Manual exclusions ───────────────────────────────────────────────────────
from scripts.utils import load_manual_exclusions

excl_eval_dir = os.path.join(DATA_DIR, "translated_terms", TERM_SLUG, "evaluation")
analysis_langs, search_terms, corrections = load_manual_exclusions(excl_eval_dir)
print(f"Manual exclusions loaded:")
print(f"  analysis_exclusion : {len(analysis_langs)} language codes  (dropped from all analysis)")
print(f"  search_exclusion   : {len(search_terms)} (language, term) pairs  (excluded from search only)")
print(f"  term_correction    : {len(corrections)} corrections")

Manual exclusions loaded:
  analysis_exclusion : 95 language codes  (dropped from all analysis)
  search_exclusion   : 383 (language, term) pairs  (excluded from search only)
  term_correction    : 94 corrections


In [ ]:
# Load the minimal-variant merged dataframe — baseline columns are identical across variants
df = load_variant_df(DATA_DIR, TERM_SLUG, "minimal")
df["language_family"] = df["language_code"].apply(get_language_family)

raw_total_langs = df["language_code"].nunique()
print(f"{TERM}: {raw_total_langs} languages loaded")

# Automated review signals from notebook 02
flags_path = os.path.join(DATA_DIR, "translated_terms", TERM_SLUG, "evaluation", "automated_review_signals.csv")
flags = read_csv_file(flags_path)
print(f"Automated review signals: {len(flags)} rows, {flags.columns.tolist()}")

# Drop analysis-excluded languages
df = df[~df["language_code"].isin(analysis_langs)].reset_index(drop=True)
total_langs = df["language_code"].nunique()
print(f"After dropping {len(analysis_langs)} analysis-excluded languages: {total_langs} languages remain.")

## 3.1 — Coverage by Source and Language Family

How many analysis-eligible languages does each prompt-invariant source reach? These sources do not use prompt variants, so this is a single count per language. Wikipedia is shown beside MT systems for comparison, but it should be read as a community-curated reference rather than a machine-translation service.


In [4]:
# Overall coverage counts
coverage_rows = []
for svc, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    n = df[col].notna().sum()
    coverage_rows.append({
        "Service": svc,
        "Covered": int(n),
        "Missing": int(total_langs - n),
        "Coverage %": round(100 * n / total_langs, 1),
    })

coverage_df = pd.DataFrame(coverage_rows).sort_values("Covered", ascending=False)
display(coverage_df)

,Service,Covered,Missing,Coverage %
1,Google Translate,229,652,26.0
3,Lingvanex,100,781,11.4
2,EasyNMT,92,789,10.4
0,Wikipedia,35,846,4.0


In [ ]:
# Coverage bar chart
bar = alt.Chart(coverage_df).mark_bar().encode(
    x=alt.X("Covered:Q", title="Languages with a translation"),
    y=alt.Y("Service:N", sort="-x", title=None),
    color=alt.Color("Service:N", legend=None),
    tooltip=["Service", "Covered", "Coverage %"],
).properties(title="Prompt-invariant source coverage", width=500, height=150)

rule = alt.Chart(pd.DataFrame({"x": [total_langs]})).mark_rule(
    color="grey", strokeDash=[4, 4]
).encode(x="x:Q")

(bar + rule)

### Prompt-Invariant Word-Count Distribution

Because these sources are prompt-invariant, word count is a service-level output profile rather than a prompt effect. This view stays here with the prompt-invariant sources; the LLM prompt-sensitive version appears in Notebook 04.


In [ ]:
length_rows = []
for service, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    vals = df[col].dropna().astype(str)
    for wc in vals.str.split().str.len():
        length_rows.append({"Service": service, "Word Count": int(wc)})

length_df = pd.DataFrame(length_rows)

bars = alt.Chart(length_df).mark_bar(opacity=0.85).encode(
    x=alt.X("Word Count:Q", bin=alt.Bin(extent=[1, 12], step=1), title="word count"),
    y=alt.Y("count():Q", title="languages"),
    color=alt.Color("Service:N", scale=alt.Scale(scheme="tableau10"), legend=None),
    tooltip=["Service:N", alt.Tooltip("count():Q", title="languages")],
).properties(width=155, height=110)

median_rule = alt.Chart(length_df).mark_rule(color="red", strokeDash=[4, 2]).encode(
    x=alt.X("median(Word Count):Q"),
    tooltip=[alt.Tooltip("median(Word Count):Q", format=".1f", title="median word count")],
)

display(
    alt.layer(bars, median_rule, data=length_df).facet(
        facet=alt.Facet("Service:N", sort=list(BASELINE_SERVICES.keys()), title=None),
        columns=2,
        title=f"Prompt-invariant translation word-count distribution — {TERM} (red line = median)",
    )
)


In [ ]:
# Coverage heatmap: service × language family
fam_rows = []
for svc, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    grp = df.groupby("language_family")[col].agg(
        total="count",
        covered=lambda x: x.notna().sum(),
    ).reset_index()
    grp["Service"] = svc
    grp["pct"] = grp["covered"] / grp["total"]
    fam_rows.append(grp)

fam_df = pd.concat(fam_rows, ignore_index=True)

# Sort families by total language count (descending)
fam_totals = df.groupby("language_family")["language_code"].nunique().sort_values(ascending=False)
fam_order = fam_totals.index.tolist()

heatmap = alt.Chart(fam_df).mark_rect().encode(
    x=alt.X("Service:N", title=None),
    y=alt.Y("language_family:N", sort=fam_order, title=None),
    color=alt.Color(
        "pct:Q",
        scale=alt.Scale(scheme="greens", domain=[0, 1]),
        title="Coverage fraction",
    ),
    tooltip=[
        alt.Tooltip("Service:N"),
        alt.Tooltip("language_family:N", title="Family"),
        alt.Tooltip("covered:Q", title="Covered"),
        alt.Tooltip("total:Q", title="Total in family"),
        alt.Tooltip("pct:Q", title="Coverage", format=".0%"),
    ],
).properties(
    title="Prompt-invariant source coverage by language family",
    width=300,
    height=700,
)
heatmap

### EasyNMT Structural Gaps by Family

EasyNMT's low overall coverage is not a random scatter of failures; it reflects the availability of Helsinki-NLP opus-mt models for each language pair. When no opus-mt model exists for a source→target language pair, EasyNMT returns a 404 and leaves the field empty. This means its gaps are structural: entire language families can be out of scope, not just individual languages.

The chart and printed list below calculate the zero-coverage families live from the current language set. The point is methodological rather than tied to a fixed count: EasyNMT can expand nominal coverage in some places, but its absence is highly patterned by training-data and model availability.


In [33]:
# EasyNMT structural gaps by language family
enmt_col = BASELINE_SERVICES["EasyNMT"]

fam_enmt = (
    df.groupby("language_family")
    .agg(
        n_langs=("language_code", "nunique"),
        n_covered=(enmt_col, lambda x: x.notna().sum()),
    )
    .reset_index()
)
fam_enmt["coverage_rate"] = fam_enmt["n_covered"] / fam_enmt["n_langs"]
fam_enmt["zero_coverage"] = fam_enmt["n_covered"] == 0
fam_enmt = fam_enmt.sort_values("coverage_rate")

enmt_bar = alt.Chart(fam_enmt).mark_bar().encode(
    y=alt.Y("language_family:N", sort="x", title=None),
    x=alt.X("coverage_rate:Q", axis=alt.Axis(format="%"), title="EasyNMT coverage rate"),
    color=alt.condition(
        alt.datum.zero_coverage,
        alt.value("#d62728"),
        alt.value("#1f77b4"),
    ),
    tooltip=[
        "language_family:N",
        alt.Tooltip("coverage_rate:Q", format=".1%"),
        alt.Tooltip("n_covered:Q", title="languages covered"),
        alt.Tooltip("n_langs:Q", title="total languages"),
    ],
).properties(width=420, height=600,
             title="EasyNMT coverage by family — red = zero (no opus-mt model for this family)")
display(enmt_bar)

zero_fams = fam_enmt[fam_enmt["zero_coverage"]]["language_family"].tolist()
print(f"Families with zero EasyNMT coverage ({len(zero_fams)}):")
for f in zero_fams:
    n = int(fam_enmt.loc[fam_enmt["language_family"]==f, "n_langs"].iloc[0])
    print(f"  {f} ({n} languages)")


alt.Chart(...)

Families with zero EasyNMT coverage (52):
  Abkhaz-Adyge (4 languages)
  Uto-Aztecan (6 languages)
  Khoe-Kwadi (1 languages)
  Kru (1 languages)
  Language isolate (8 languages)
  Maban (1 languages)
  Mande (9 languages)
  Mayan (2 languages)
  Mongolic-Khitan (2 languages)
  Muskogean (4 languages)
  Nakh-Daghestanian (8 languages)
  Nilo-Saharan languages (3 languages)
  North American Indian languages (5 languages)
  Nubian (1 languages)
  Japonic languages (1 languages)
  Nuclear-Macro-Je (1 languages)
  Pama-Nyungan (1 languages)
  Quechuan (1 languages)
  Saharan (2 languages)
  Salishan (4 languages)
  Sign languages (1 languages)
  Siouan (3 languages)
  Songhay (4 languages)
  South American Indian languages (4 languages)
  Tai-Kadai (1 languages)
  Tai-Kadai languages (8 languages)
  Tungusic (2 languages)
  Tupian (1 languages)
  Turkic (19 languages)
  Otomanguean (1 languages)
  Iroquoian (4 languages)
  Kartvelian (3 languages)
  Indo-European (1 languages)
  Afro-Asiat

### Source Overlap: How Many Languages Have Prompt-Invariant Coverage?

This overlap view shows how many languages have 0, 1, 2, 3, or all 4 prompt-invariant sources covering them. It gives a live measure of how much cross-source validation is available before LLM prompting enters the pipeline. Wikipedia is community-curated rather than MT, so its agreement carries different epistemic weight from MT-MT agreement.


In [ ]:
# Source overlap: how many prompt-invariant sources cover each language?
df["n_baseline"] = sum(
    df[col].notna().astype(int)
    for col in BASELINE_SERVICES.values()
    if col in df.columns
)

overlap_counts = df["n_baseline"].value_counts().sort_index().reset_index()
overlap_counts.columns = ["n_services", "n_languages"]
overlap_counts["label"] = overlap_counts["n_services"].map({
    0: "No prompt-invariant coverage",
    1: "1 source",
    2: "2 sources",
    3: "3 sources",
    4: "All 4 sources",
})

print(f"Prompt-invariant source overlap across {total_langs:,} analysis-eligible languages:")
for _, r in overlap_counts.iterrows():
    pct = r["n_languages"] / total_langs * 100
    print(f"  {r['label']:25s}: {r['n_languages']:4d}  ({pct:.1f}%)")

overlap_bar = alt.Chart(overlap_counts).mark_bar().encode(
    x=alt.X("label:N",
            sort=["No prompt-invariant coverage","1 source","2 sources","3 sources","All 4 sources"],
            title=None),
    y=alt.Y("n_languages:Q", title="languages"),
    color=alt.Color("n_services:O", scale=alt.Scale(scheme="blues"), legend=None),
    tooltip=["label:N", "n_languages:Q"],
).properties(width=340, height=200, title="How many prompt-invariant sources cover each language?")
display(overlap_bar)

zero_by_fam = (
    df[df["n_baseline"] == 0]
    .groupby("language_family")["language_code"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="n_zero")
)
print(f"\nZero prompt-invariant coverage languages by family:")
for _, r in zero_by_fam.iterrows():
    total_fam = df[df["language_family"] == r["language_family"]]["language_code"].nunique()
    pct = r["n_zero"] / total_fam * 100
    print(f"  {r['language_family']:45s}: {r['n_zero']:3d}/{total_fam:3d} ({pct:.0f}%)")


### Pairwise Source Overlap — Verification vs Expansion

The overlap chart above shows *how many* sources cover each language; the natural follow-up is *which* sources cover which others. The methodological question this answers is concrete: **does adding each non-Google-Translate baseline expand pipeline coverage, or does it largely verify what Google Translate already returned?** A source that overlaps almost entirely with Google Translate is providing cross-source validation but no new language coverage; a source with substantial unique coverage is genuinely expanding the dataset.

Below: pairwise overlap matrix, then a focused breakdown of each non-GT baseline's coverage split into the share that *verifies* Google Translate (overlap) versus the share that *expands* beyond it (unique to that service).

In [10]:
# Build per-service coverage sets (languages with a non-null translation)
cov_sets = {
    label: set(df.loc[df[col].notna(), 'language_code'])
    for label, col in BASELINE_SERVICES.items()
    if col in df.columns
}
svc_labels = list(cov_sets)

# Pairwise intersection matrix (counts)
overlap_mat = pd.DataFrame(
    [[len(cov_sets[a] & cov_sets[b]) for b in svc_labels] for a in svc_labels],
    index=svc_labels, columns=svc_labels,
)
print('── Pairwise overlap (number of languages covered by BOTH row and column) ──')
print(overlap_mat.to_string())

# % of ROW service's coverage that is also covered by COLUMN service
overlap_pct = pd.DataFrame(
    [[round(100 * len(cov_sets[a] & cov_sets[b]) / max(1, len(cov_sets[a])), 1)
      for b in svc_labels] for a in svc_labels],
    index=svc_labels, columns=svc_labels,
)
print('\n── % of ROW service\'s coverage that COLUMN service also covers ──')
print('(diagonal = 100% by definition; row "Wikipedia", column "GT" of 100% means every Wikipedia-covered language is also covered by GT)\n')
print(overlap_pct.to_string())

── Pairwise overlap (number of languages covered by BOTH row and column) ──
                  Wikipedia  Google Translate  EasyNMT  Lingvanex
Wikipedia                35                35       22         33
Google Translate         35               229       79        100
EasyNMT                  22                79       92         46
Lingvanex                33               100       46        100

── % of ROW service's coverage that COLUMN service also covers ──
(diagonal = 100% by definition; row "Wikipedia", column "GT" of 100% means every Wikipedia-covered language is also covered by GT)

                  Wikipedia  Google Translate  EasyNMT  Lingvanex
Wikipedia             100.0             100.0     62.9       94.3
Google Translate       15.3             100.0     34.5       43.7
EasyNMT                23.9              85.9    100.0       50.0
Lingvanex              33.0             100.0     46.0      100.0


In [ ]:
# Direct verification-vs-expansion breakdown using Google Translate as the reference,
# since it has the broadest coverage (it's the de-facto MT baseline this paper benchmarks against).
if 'Google Translate' not in cov_sets:
    raise RuntimeError('Google Translate coverage set missing — check BASELINE_SERVICES.')
gt = cov_sets['Google Translate']

ve_rows = []
for svc in svc_labels:
    if svc == 'Google Translate': continue
    s = cov_sets[svc]
    overlap = s & gt
    only_this = s - gt
    ve_rows.append({
        'service': svc,
        'total_covered': len(s),
        'shared_with_GT': len(overlap),
        'unique_vs_GT':   len(only_this),
        'verification_pct': round(100 * len(overlap) / max(1, len(s)), 1),
        'expansion_pct':    round(100 * len(only_this) / max(1, len(s)), 1),
    })
ve = pd.DataFrame(ve_rows)
print('── Verification vs Expansion (Google Translate as reference) ──\n')
print(ve.to_string(index=False))

# Also: how many languages are uniquely covered by ONE service (no other baseline covers them)?
uniq_rows = []
for label in svc_labels:
    others = set().union(*(cov_sets[k] for k in svc_labels if k != label))
    uniq = cov_sets[label] - others
    uniq_rows.append({
        'service': label,
        'total_covered': len(cov_sets[label]),
        'unique_to_this_service': len(uniq),
        'unique_pct': round(100 * len(uniq) / max(1, len(cov_sets[label])), 1),
    })
uniq_df = pd.DataFrame(uniq_rows).sort_values('unique_to_this_service', ascending=False)
print('\n── Languages uniquely covered by exactly one prompt-invariant source ──')
print('(How much coverage *only* this source provides — covered by no other prompt-invariant source)\n')
print(uniq_df.to_string(index=False))

In [12]:
# Visualisation: stacked bar showing verification vs expansion per non-GT service
ve_long = ve.melt(
    id_vars='service',
    value_vars=['shared_with_GT', 'unique_vs_GT'],
    var_name='kind', value_name='n_languages',
)
ve_long['kind'] = ve_long['kind'].map({
    'shared_with_GT': 'Shared with Google Translate (verification)',
    'unique_vs_GT':   'Not in Google Translate (expansion)',
})

bar = alt.Chart(ve_long).mark_bar().encode(
    y=alt.Y('service:N', sort=alt.SortField('n_languages', order='descending'),
            title=None),
    x=alt.X('n_languages:Q', title='languages covered'),
    color=alt.Color('kind:N',
                    scale=alt.Scale(
                        domain=['Shared with Google Translate (verification)',
                                'Not in Google Translate (expansion)'],
                        range=['#9ecae1', '#1976d2']),
                    legend=alt.Legend(orient='bottom', title=None)),
    tooltip=['service:N', 'kind:N', 'n_languages:Q'],
).properties(
    width=440, height=160,
    title=alt.Title(
        'How much does each non-Google baseline expand vs verify GT coverage?',
        subtitle='Dark = languages this service adds beyond GT; light = languages GT already covers',
    ),
)
labels = bar.mark_text(align='center', dx=0, fontSize=10, color='white').encode(
    text='n_languages:Q',
)
(bar + labels)

alt.LayerChart(...)

#### What does EasyNMT actually add?

EasyNMT's expansion beyond the other prompt-invariant sources is worth inspecting individually because raw coverage can overstate usable coverage. The list below shows languages where EasyNMT is the only prompt-invariant source with an output for `Digital Humanities`.

Several patterns are worth noting:

- **Helsinki-NLP / JW.org training signal**: EasyNMT relies on OPUS-MT models trained partly on Jehovah's Witnesses parallel corpora (the JW.org translation effort covers many languages with sparse other digital text). Outputs that begin with `O Afendeli va Yehova...` (Umbundu) or have a strong religious-text register (Lozi, Tuvalu, Lunda) reflect this training distribution.
- **Some outputs are unusable as translations**: case-only headers and clear non-translation strings may nominally count as coverage but should not be treated as usable search terms.
- **Some outputs may be genuine translations** and deserve manual inspection rather than automatic rejection.

The methodological reading: EasyNMT can expand nominal coverage, but the usable expansion is smaller once the automated review signals and manual review are applied.


In [ ]:
# Inspect the EasyNMT-unique languages individually
gt_set    = cov_sets['Google Translate']
wiki_set  = cov_sets.get('Wikipedia', set())
lingv_set = cov_sets.get('Lingvanex', set())
_others    = gt_set | wiki_set | lingv_set
only_enmt = cov_sets['EasyNMT'] - _others

enmt_unique_df = (
    df[df['language_code'].isin(only_enmt)]
    [['language_code', 'language_name', 'language_family', 'enmt_translated_term']]
    .sort_values('language_name')
    .rename(columns={'enmt_translated_term': 'EasyNMT output for Digital Humanities'})
)
print(f'Languages where EasyNMT is the ONLY prompt-invariant source with coverage: {len(enmt_unique_df)}\n')
# Truncate long outputs for display
disp = enmt_unique_df.copy()
disp['EasyNMT output for Digital Humanities'] = disp['EasyNMT output for Digital Humanities'].astype(str).str.slice(0, 70)
print(disp.to_string(index=False))

In [14]:
ve_rows

[{'service': 'Wikipedia',
  'total_covered': 35,
  'shared_with_GT': 35,
  'unique_vs_GT': 0,
  'verification_pct': 100.0,
  'expansion_pct': 0.0},
 {'service': 'EasyNMT',
  'total_covered': 92,
  'shared_with_GT': 79,
  'unique_vs_GT': 13,
  'verification_pct': 85.9,
  'expansion_pct': 14.1},
 {'service': 'Lingvanex',
  'total_covered': 100,
  'shared_with_GT': 100,
  'unique_vs_GT': 0,
  'verification_pct': 100.0,
  'expansion_pct': 0.0}]

### Official Support vs Actual Collection

The coverage charts above show what our pipeline *actually collected* from Google Translate. But Google publishes two lists of *officially supported* languages — one for their standard NMT endpoint (~190 languages), one for the higher-quality Translation LLM endpoint (~115 languages). Cross-referencing those lists against what we collected surfaces two methodologically interesting gaps that the rest of this paper's MT-baselines argument depends on.

**Two questions:**

1. **For the languages Google officially supports** — did our pipeline collect a real translation in every case? If not, the gap between *advertised* support and *delivered* output is itself a finding (network errors, rate limits, or silent fallback to the source term).
2. **For the languages Google does NOT officially support** — did the API return *something* anyway? Google's API does not strictly refuse unsupported languages; it often returns plausible-looking output (the source term unchanged, a transliteration, or a near-language calque). This is the *implicit* coverage that doesn't appear in any official list, and characterising it is central to the paper's argument about MT coverage gaps.

The `google_nmt_supported` and `google_translation_llm_supported` columns added by `add_service_language_codes()` in `generate_language_codes.py` allow this cross-reference directly. Below, *real translation* means: non-empty, not an exact pass-through of `'Digital Humanities'`, not source-term leakage (e.g., contains `Digital Humanities` as a substring), and not a placeholder phrase like `'Translation not available'`.

In [15]:
# Load official-support columns from the metadata table and join to the working df
meta = pd.read_csv(
    os.path.join(DATA_DIR, 'metadata_files', 'language_codes_comprehensive.csv'),
    converters={'language_code': str},
)
support_cols = ['google_nmt_supported', 'google_translation_llm_supported']
assert all(c in meta.columns for c in support_cols), (
    'Run scripts/experiment/generate_language_codes.py or apply '
    'add_service_language_codes() to populate the support columns.'
)

df_sup = df.merge(meta[['language_code'] + support_cols], on='language_code', how='left')
for c in support_cols:
    df_sup[c] = df_sup[c].fillna(False).astype(bool)

print(f'Pipeline languages: {len(df_sup)}')
print(f'  Google NMT officially supported       : {df_sup["google_nmt_supported"].sum()}')
print(f'  Google Translation LLM officially supp.: {df_sup["google_translation_llm_supported"].sum()}')

Pipeline languages: 786
  Google NMT officially supported       : 172
  Google Translation LLM officially supp.: 80


In [16]:
# Classify each Google Translate output into 'real translation' or 'not real'
PLACEHOLDER_GT = {'translation not available', 'no direct translation', 'untranslatable',
                   'no translation', 'not available', 'unknown'}

def is_real_translation(val):
    if not isinstance(val, str): return False
    v = val.strip()
    if not v or v.lower() in {'nan', 'none', ''}: return False
    if v.strip().lower() == TERM.lower(): return False                # exact pass-through
    if has_source_leakage(v, TERM): return False                      # source-term leakage
    if v.strip().lower() in PLACEHOLDER_GT: return False             # placeholder
    if is_placeholder_term(v): return False                           # placeholder regex from classifier
    return True

df_sup['gt_real_translation'] = df_sup['gt_translated_term'].apply(is_real_translation)

# Build the 2×2 quadrant for Google NMT
def _quadrant(df_in, support_col, real_col):
    return pd.DataFrame({
        'support': ['Officially supported', 'Officially supported',
                    'NOT officially supported', 'NOT officially supported'],
        'outcome': ['Got a real translation', 'Got nothing/junk',
                    'Got a real translation', 'Got nothing/junk'],
        'count': [
            int(( df_in[support_col] &  df_in[real_col]).sum()),
            int(( df_in[support_col] & ~df_in[real_col]).sum()),
            int((~df_in[support_col] &  df_in[real_col]).sum()),
            int((~df_in[support_col] & ~df_in[real_col]).sum()),
        ],
    })

nmt_quadrant = _quadrant(df_sup, 'google_nmt_supported', 'gt_real_translation')
nmt_quadrant['pct'] = (nmt_quadrant['count'] / len(df_sup) * 100).round(1)
print('── Google NMT: officially supported × actually delivered ──')
print(nmt_quadrant.to_string(index=False))

── Google NMT: officially supported × actually delivered ──
                 support                outcome  count  pct
    Officially supported Got a real translation    159 20.2
    Officially supported       Got nothing/junk     13  1.7
NOT officially supported Got a real translation     53  6.7
NOT officially supported       Got nothing/junk    561 71.4


In [17]:
# Visualise the 2×2 as a heatmap
quadrant_labels = {
    ('Officially supported', 'Got a real translation'):     '✓ advertised + delivered',
    ('Officially supported', 'Got nothing/junk'):            '✗ advertised but failed',
    ('NOT officially supported', 'Got a real translation'):  '⊕ delivered without support',
    ('NOT officially supported', 'Got nothing/junk'):        '∅ expected absence',
}
nmt_quadrant['label'] = nmt_quadrant.apply(
    lambda r: quadrant_labels.get((r['support'], r['outcome']), ''), axis=1,
)
nmt_quadrant['display'] = nmt_quadrant.apply(
    lambda r: f"{r['count']} ({r['pct']:.1f}%)", axis=1,
)

heat = alt.Chart(nmt_quadrant).mark_rect().encode(
    x=alt.X('outcome:N', sort=['Got a real translation', 'Got nothing/junk'],
            title='Actual collection outcome',
            axis=alt.Axis(labelAngle=0)),
    y=alt.Y('support:N', sort=['Officially supported', 'NOT officially supported'],
            title='Google NMT advertised support'),
    color=alt.Color('count:Q', scale=alt.Scale(scheme='blues'), legend=None),
    tooltip=['support:N', 'outcome:N', 'count:Q', 'pct:Q', 'label:N'],
).properties(width=320, height=180,
             title=alt.Title('Google NMT: advertised vs delivered',
                             subtitle='Top-left = ideal; top-right = silent failures; bottom-left = implicit coverage'))
labels = heat.mark_text(fontSize=12, fontWeight='bold').encode(
    text='display:N',
    color=alt.condition('datum.count > 400', alt.value('white'), alt.value('black')),
)
(heat + labels)

alt.LayerChart(...)

In [18]:
# Same quadrant for Google's Translation LLM endpoint
llm_quadrant = _quadrant(df_sup, 'google_translation_llm_supported', 'gt_real_translation')
llm_quadrant['pct'] = (llm_quadrant['count'] / len(df_sup) * 100).round(1)
print('── Google Translation LLM: officially supported × actually delivered ──')
print(llm_quadrant.to_string(index=False))

── Google Translation LLM: officially supported × actually delivered ──
                 support                outcome  count  pct
    Officially supported Got a real translation     75  9.5
    Officially supported       Got nothing/junk      5  0.6
NOT officially supported Got a real translation    137 17.4
NOT officially supported       Got nothing/junk    569 72.4


In [19]:
# Spot-check samples for the two unexpected buckets
advertised_failed = df_sup[df_sup['google_nmt_supported'] & ~df_sup['gt_real_translation']]
implicit_coverage = df_sup[~df_sup['google_nmt_supported'] &  df_sup['gt_real_translation']]

print(f'── Advertised but failed: {len(advertised_failed)} languages ──')
print('(Google officially supports these in NMT but our pipeline did not collect a real translation)\n')
if len(advertised_failed):
    print(advertised_failed[['language_code', 'language_name', 'gt_translated_term']]
          .head(15).to_string(index=False))

print(f'\n── Implicit coverage: {len(implicit_coverage)} languages ──')
print('(Not in any official Google NMT support list, but the API returned a real-looking translation anyway)\n')
if len(implicit_coverage):
    print(implicit_coverage[['language_code', 'language_name', 'gt_translated_term']]
          .head(15).to_string(index=False))

── Advertised but failed: 13 languages ──
(Google officially supports these in NMT but our pipeline did not collect a real translation)

language_code language_name                      gt_translated_term
           nr South Ndebele                    I-Digital Humanities
          pag    Pangasinan                      Digital Humanities
          pam   Kapampangan                      Digital Humanities
      map-bms    Banyumasan                                     NaN
          lus          Mizo Digital Humanities hmanga thil tih a ni
           tl       Tagalog                      Digital Humanities
          ach         Acoli                      Digital Humanities
          bik         Bikol                      Digital Humanities
          fil      Filipino                      Digital Humanities
           lb Luxembourgish                      Digital Humanities
          hil    Hiligaynon                      Digital Humanities
           zu          Zulu                    

## 3.2 — Pass-Through Rate: Services that Echo the Source Term

A translation that returns "Digital Humanities" unchanged is not really a translation; it is a coverage failure that looks like data. The rate at which each service does this tells us how much of the nominal coverage is genuine.

In [ ]:
pass_rows = []
for svc, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    vals = df[col].dropna().astype(str)
    covered = len(vals)
    exact = (vals.str.strip().str.lower() == TERM.lower()).sum()
    contains = vals.str.contains(TERM, case=False, na=False).sum()
    source_leak = sum(has_source_leakage(v, TERM) for v in vals)
    pass_rows.append({
        "Service": svc,
        "Covered": covered,
        "Exact pass-through": int(exact),
        "Contains source term": int(contains),
        "Source leakage (incl. initials)": int(source_leak),
        "Pass-through %": round(100 * exact / covered, 1) if covered else 0,
        "Source leakage %": round(100 * source_leak / covered, 1) if covered else 0,
    })

pass_df = pd.DataFrame(pass_rows)
display(pass_df)

In [ ]:
# Stacked bar: genuine translation vs pass-through vs source-leakage
stacked_rows = []
for _, row in pass_df.iterrows():
    exact = row["Exact pass-through"]
    leak_extra = row["Source leakage (incl. initials)"] - exact
    genuine = row["Covered"] - row["Source leakage (incl. initials)"]
    stacked_rows += [
        {"Service": row["Service"], "Category": "Genuine translation", "Count": genuine},
        {"Service": row["Service"], "Category": "Contains DH (non-exact)", "Count": max(0, leak_extra)},
        {"Service": row["Service"], "Category": "Exact pass-through", "Count": exact},
    ]

stacked_df = pd.DataFrame(stacked_rows)

stacked_bar = alt.Chart(stacked_df).mark_bar().encode(
    x=alt.X("sum(Count):Q", title="Languages"),
    y=alt.Y("Service:N", sort="-x", title=None),
    color=alt.Color(
        "Category:N",
        scale=alt.Scale(
            domain=["Genuine translation", "Contains DH (non-exact)", "Exact pass-through"],
            range=["#4c9b5e", "#f0a830", "#c9413a"],
        ),
    ),
    tooltip=["Service", "Category", "sum(Count):Q"],
    order=alt.Order("Category:N", sort="ascending"),
).properties(title="Prompt-invariant source breakdown: genuine vs pass-through", width=500, height=150)

stacked_bar

In [22]:
# Which languages are pass-throughs? Show the top offenders per service
for svc, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    passthrough_mask = df[col].str.strip().str.lower() == TERM.lower()
    rows = df.loc[passthrough_mask, ["language_code", "language_name", "language_family", col]]
    if len(rows):
        print(f"\n{svc} — {len(rows)} exact pass-throughs:")
        display(rows.reset_index(drop=True))
    else:
        print(f"\n{svc} — no exact pass-throughs")


Wikipedia — 2 exact pass-throughs:


,language_code,language_name,language_family,wikipedia_translated_term
0,de,German,Indo-European languages,Digital Humanities
1,en,English,Indo-European languages,digital humanities



Google Translate — 14 exact pass-throughs:


,language_code,language_name,language_family,gt_translated_term
0,pag,Pangasinan,Austronesian languages,Digital Humanities
1,pam,Kapampangan,Austronesian languages,Digital Humanities
2,mh,Marshallese,Austronesian languages,Digital Humanities
3,tl,Tagalog,Austronesian languages,Digital Humanities
4,sus,Susu,Mande,Digital Humanities
5,ch,Chamorro,Austronesian languages,Digital Humanities
6,ach,Acoli,Nilotic,Digital Humanities
7,bik,Bikol,Austronesian languages,Digital Humanities
8,bcl,Bikol,Austronesian languages,Digital Humanities
9,fil,Filipino,Austronesian languages,Digital Humanities



EasyNMT — 1 exact pass-throughs:


,language_code,language_name,language_family,enmt_translated_term
0,it,Italian,Indo-European languages,Digital Humanities



Lingvanex — 17 exact pass-throughs:


,language_code,language_name,language_family,lingvanex_translated_term
0,no,Norwegian,Indo-European languages,Digital Humanities
1,ny,Chichewa; Chewa; Nyanja,Atlantic-Congo,Digital Humanities
2,nl,Dutch,Indo-European languages,Digital Humanities
3,mg,Malagasy,Austronesian languages,Digital Humanities
4,yo,Yoruba,Atlantic-Congo,Digital Humanities
5,sn,Shona,Atlantic-Congo,Digital Humanities
6,tl,Tagalog,Austronesian languages,Digital Humanities
7,st,Southern Sotho,Atlantic-Congo,Digital Humanities
8,lo,Lao,Tai-Kadai languages,Digital Humanities
9,bs,Bosnian,Indo-European languages,Digital Humanities


## 3.3 — Automated Review Signals per Prompt-Invariant Source

Run each prompt-invariant translation through the translation-level parts of `translation_classifier.py`. These sources do not produce rationale text or prompt variants, so rationale-level signals and prompt-level disagreements do not apply here. The purpose is to ask which source outputs trigger automated review signals before manual review or downstream filtering.


In [ ]:
CHECKERS = {
    "placeholder": is_placeholder_term,
    "repetition_loop": is_repetition_loop,
    "extreme_length": has_extreme_term_length,
    "unicode_escape": has_unicode_escape,
}

flag_rows = []
for svc, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    vals = df[col].dropna().astype(str)
    covered = len(vals)
    row = {"Service": svc, "Covered": covered}
    for flag_name, fn in CHECKERS.items():
        count = sum(fn(v) for v in vals)
        row[flag_name] = count
        row[f"{flag_name}_pct"] = round(100 * count / covered, 1) if covered else 0

    # mixed-script / stripped via curate_translation
    nulled = stripped = 0
    for v in vals:
        _, action = curate_translation(v)
        if action == "nulled":
            nulled += 1
        elif action == "stripped":
            stripped += 1
    row["mixed_script"] = nulled
    row["mixed_script_pct"] = round(100 * nulled / covered, 1) if covered else 0
    row["romanization_stripped"] = stripped
    row["romanization_stripped_pct"] = round(100 * stripped / covered, 1) if covered else 0

    flag_rows.append(row)

flag_df = pd.DataFrame(flag_rows)

# Display count table
count_cols = ["Service", "Covered", "placeholder", "repetition_loop", "extreme_length",
              "unicode_escape", "mixed_script", "romanization_stripped"]
display(flag_df[count_cols])

In [ ]:
# Flag rate chart
flag_long_rows = []
flag_display = {
    "placeholder": "Placeholder/refusal",
    "repetition_loop": "Repetition loop",
    "extreme_length": "Extreme length (>100 chars)",
    "unicode_escape": "Unicode escape (\\uXXXX)",
    "mixed_script": "Mixed script (nulled)",
    "romanization_stripped": "Romanization stripped",
}
for _, row in flag_df.iterrows():
    for flag, label in flag_display.items():
        pct_col = f"{flag}_pct"
        if pct_col in flag_df.columns:
            flag_long_rows.append({
                "Service": row["Service"],
                "Flag": label,
                "Rate": row[pct_col],
                "Count": int(row[flag]),
            })

flag_long_df = pd.DataFrame(flag_long_rows)

# Only show flags that actually fired
nonzero_flags = flag_long_df.groupby("Flag")["Count"].sum()
active_flags = nonzero_flags[nonzero_flags > 0].index.tolist()

if active_flags:
    active_df = flag_long_df[flag_long_df["Flag"].isin(active_flags)]
    flag_chart = alt.Chart(active_df).mark_bar().encode(
        x=alt.X("Rate:Q", title="% of covered translations"),
        y=alt.Y("Service:N", title=None),
        color=alt.Color("Service:N", legend=None),
        row=alt.Row("Flag:N", title=None),
        tooltip=["Service", "Flag", "Count", alt.Tooltip("Rate:Q", format=".1f", title="%")],
    ).properties(width=400, height=80, title="Prompt-invariant automated review signal rates (% of covered languages)")
    display(flag_chart)
else:
    print("No automated review signals fired for any prompt-invariant source.")

In [ ]:
# Show offending rows for any flagged prompt-invariant translations
for svc, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    rows_out = []
    for _, row in df[["language_code", "language_name", "language_family", col]].dropna(subset=[col]).iterrows():
        v = str(row[col])
        flags_fired = []
        for flag_name, fn in CHECKERS.items():
            if fn(v):
                flags_fired.append(flag_name)
        _, action = curate_translation(v)
        if action in ("nulled", "stripped"):
            flags_fired.append(action)
        if flags_fired:
            rows_out.append({
                "language_code": row["language_code"],
                "language_name": row["language_name"],
                "language_family": row["language_family"],
                "translation": v[:120],
                "flags": ", ".join(flags_fired),
            })
    if rows_out:
        print(f"\n{svc} — {len(rows_out)} flagged translations:")
        display(pd.DataFrame(rows_out))
    else:
        print(f"\n{svc} — no automated review signals")

## 3.4 — Mixed-Script Output

Do any prompt-invariant sources produce mixed-script output (characters from two or more writing systems interleaved in a single term)? This is relatively rare in MT systems compared to LLMs, but EasyNMT's open-domain opus-mt models can produce unexpected romanization helpers.

In [ ]:
from scripts.utils import detect_dominant_script

script_rows = []
for svc, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    for _, row in df[["language_code", "language_name", "language_family", col]].dropna(subset=[col]).iterrows():
        v = str(row[col])
        script = detect_dominant_script(v)
        _, action = curate_translation(v)
        script_rows.append({
            "Service": svc,
            "language_code": row["language_code"],
            "language_name": row["language_name"],
            "language_family": row["language_family"],
            "translation": v[:80],
            "dominant_script": script,
            "clean_action": action,
        })

script_df = pd.DataFrame(script_rows)

# Script distribution per service
script_dist = (
    script_df.groupby(["Service", "dominant_script"])
    .size()
    .reset_index(name="count")
)
script_chart = alt.Chart(script_dist).mark_bar().encode(
    x=alt.X("count:Q", title="Translations"),
    y=alt.Y("dominant_script:N", title=None, sort="-x"),
    color=alt.Color("Service:N"),
    tooltip=["Service", "dominant_script", "count"],
).properties(
    title="Script distribution of prompt-invariant translations",
    width=400, height=250
)
script_chart

In [ ]:
# Translations that needed cleaning (stripped or nulled)
needing_clean = script_df[script_df["clean_action"].isin(["stripped", "nulled"])]
if len(needing_clean):
    print(f"{len(needing_clean)} prompt-invariant translations needed cleaning:")
    display(needing_clean[["Service", "language_name", "translation", "clean_action"]].reset_index(drop=True))
else:
    print("No prompt-invariant translations required cleaning — all pass through unchanged or as-is.")

## 3.5 — Filter Funnel: Full, Quality-Filtered, and Search-Ready Views

This section is a descriptive funnel, not the Notebook 04 Tier 1 service-exploration policy. It shows how much nominal prompt-invariant coverage remains after two increasingly strict filters:

- **Full / nominal coverage**: any non-null output counts as coverage.
- **Quality-filtered view**: likely term failures are removed: mixed-script nulls, placeholder/refusal terms, repetition loops, extreme length, and literal Unicode escapes.
- **Search-ready view**: source-term leakage is also removed, because exact `Digital Humanities` echoes are not useful search terms even though they remain informative evidence of service behavior.

Pass-throughs are therefore retained as data in exploratory analysis but excluded from the search-ready view.


In [ ]:
# Quality-filtered view: remove likely term failures
tier1_rows = []
for svc, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    vals = df[col].dropna().astype(str)
    covered = len(vals)
    usable = 0
    excluded = 0
    for v in vals:
        _, action = curate_translation(v)
        is_bad = (
            action in ("nulled", "placeholder")
            or is_repetition_loop(v)
            or has_extreme_term_length(v)
            or has_unicode_escape(v)
        )
        if is_bad:
            excluded += 1
        else:
            usable += 1
    tier1_rows.append({
        "Service": svc,
        "Covered": covered,
        "Excluded (quality filters)": excluded,
        "Usable": usable,
        "Usable %": round(100 * usable / covered, 1) if covered else 0,
    })

tier1_df = pd.DataFrame(tier1_rows)
display(tier1_df)

In [ ]:
# Search-ready view: additionally exclude pass-throughs (source term in translation)
tier2_rows = []
for svc, col in BASELINE_SERVICES.items():
    if col not in df.columns:
        continue
    vals = df[col].dropna().astype(str)
    covered = len(vals)
    usable = 0
    excluded = 0
    for v in vals:
        _, action = curate_translation(v)
        is_bad = (
            action in ("nulled", "placeholder")
            or is_repetition_loop(v)
            or has_extreme_term_length(v)
            or has_unicode_escape(v)
            or has_source_leakage(v, TERM)
        )
        if is_bad:
            excluded += 1
        else:
            usable += 1
    tier2_rows.append({
        "Service": svc,
        "Covered": covered,
        "Excluded (quality + source leakage)": excluded,
        "Usable": usable,
        "Usable %": round(100 * usable / covered, 1) if covered else 0,
    })

tier2_df = pd.DataFrame(tier2_rows)
print("Search-ready view (quality filters + source leakage exclusion):")
display(tier2_df)

In [ ]:
# Comparison: nominal coverage → quality-filtered → search-ready
compare_rows = []
for (_, r1), (_, r2) in zip(tier1_df.iterrows(), tier2_df.iterrows()):
    svc = r1["Service"]
    compare_rows += [
        {"Service": svc, "Stage": "Nominal coverage", "Count": r1["Covered"]},
        {"Service": svc, "Stage": "Quality-filtered", "Count": r1["Usable"]},
        {"Service": svc, "Stage": "Search-ready", "Count": r2["Usable"]},
    ]

compare_df = pd.DataFrame(compare_rows)
stage_order = ["Nominal coverage", "Quality-filtered", "Search-ready"]

funnel = alt.Chart(compare_df).mark_bar().encode(
    x=alt.X("Count:Q", title="Languages"),
    y=alt.Y("Stage:N", sort=stage_order, title=None),
    color=alt.Color(
        "Stage:N",
        sort=stage_order,
        scale=alt.Scale(scheme="blues"),
    ),
    row=alt.Row("Service:N", title=None),
    tooltip=["Service", "Stage", "Count"],
).properties(width=400, height=60, title="Prompt-invariant coverage funnel by filter view")

funnel